# SpineCounts

Computes total spine counts and apical dendrite lengths for Output, MG, and SG cells within a densely-reconstructed EM subvolume. Results are saved to `EM_data_published/data_processed_published/df_spine_counts.csv` for use in `Analyses_published.ipynb` (Figure 2A, 2B).

## Requirements
- CloudVolume and Igneous must be installed: `pip install cloud-volume igneous`
- **Mac/Linux only** — CloudVolume does not support Windows
- Before running the CloudVolume cells, start the local Igneous server in a terminal:
  ```bash
  cd /path/to/EM_data_published/data_VAST/volume_subsample_sg-mg-out_ratio/
  igneous view precomputed_subvolume-apical-revision --port 8001
  ```
- This notebook must be launched from `efish_em/Notebooks_Jupyter/`

## Output
Saves `EM_data_published/data_processed_published/df_spine_counts.csv`.
This file is loaded by `Analyses_published.ipynb` in the Spine Counts section.

In [ ]:
import sys
from pathlib import Path
DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))
import AnalysisCode as efish

import numpy as np
import pandas as pd
import networkx as nx
from copy import deepcopy
from tqdm import tqdm
from cloudvolume import CloudVolume
from neuprint import skeleton as npskel

DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'
folder_path = DATA_ROOT / 'data_VAST/volume_subsample_sg-mg-out_ratio'

In [ ]:
df_type = pd.read_csv(DATA_ROOT / 'data_processed_published/df_type_auto_typed.csv')

In [ ]:
# Load segment assignment table (contains cell IDs and types for subvolume segments)
df_segments = pd.read_csv(folder_path / 'df_segments_assigned.csv')

print(f"Output cells (LG + LF) in subvolume: {df_segments[df_segments['cell_type'].isin(['lg','lf'])]['cell_id'].nunique()}")
print(f"MG cells in subvolume: {df_segments[df_segments['cell_type'].isin(['mg1','mg2'])]['cell_id'].nunique()}")
print(f"SG cells in subvolume: {df_segments[df_segments['cell_type'].isin(['sg1','sg2'])]['cell_id'].nunique()}")

In [ ]:
def create_bins(boundaries):
    bins = []
    for i in range(len(boundaries) - 1):
        bins.append((boundaries[i], boundaries[i + 1]))
    return bins

In [ ]:
# Connect to local Igneous server
# Start the server first in a terminal (see notebook header for instructions):
#   cd /path/to/EM_data_published/data_VAST/volume_subsample_sg-mg-out_ratio/
#   igneous view precomputed_subvolume-apical-revision --port 8001

vol = CloudVolume('precomputed://http://localhost:8001', progress=False)
print(f"CloudVolume connected. Bounds: {vol.bounds}")

In [ ]:
segIDs_all = vol.unique(bbox=vol.bounds) - set([0])  # remove background segment ID 0
print(f"Total segments in volume: {len(segIDs_all)}")

In [ ]:
seglist = list(segIDs_all)

In [ ]:
df_result = pd.DataFrame()

mol_bounds = [1000, 3000, 5000, 7000, 9000, 12000, 14000]
bins = create_bins(mol_bounds)

for seg in tqdm(seglist, desc='Processing segments'):
    skel = vol.skeleton.get(seg)
    swc = skel.to_swc()
    df = npskel.skeleton_swc_to_df(swc)

    # Transform positions to 16nm segmentation scale
    df['y'] = -(df['y'] / 16 - 16210)  # offset y like spine density analysis
    df['x'] = df['x'] / 16
    df['z'] = df['z'] / 30
    vox_to_nm = 16  # transform distance back to nm

    d_total = []
    for i, b in enumerate(bins):
        mask = (df['y'] > b[0]) & (df['y'] < b[1])
        G = npskel.skeleton_df_to_nx(df[mask], with_attributes=True, directed=True,
                                      with_distances=True, virtual_roots=True)
        distance = nx.get_edge_attributes(G, 'distance')
        d_total.append(
            sum(filter(lambda x: x != float('inf'), np.asarray(list(distance.values())))) * vox_to_nm / 1000
        )

    df_ = pd.DataFrame({
        'depth': bins,
        'labels': ['d0', 'd1', 'd2', 'd3', 'd4', 'd5'],
        'dx_total': d_total
    })
    df_['seg'] = seg
    df_['ctype'] = df_type[df_type['id'].isin([seg])]['cell_type'].values[0]
    df_result = pd.concat([df_result, df_], ignore_index=True)

print(f"Computed dendritic lengths for {df_result['seg'].nunique()} segments")
print(df_result.head())

In [ ]:
df_result.loc[df_result['ctype'].isin(['sg1', 'sg2']), 'class'] = 'sg'
df_result.loc[df_result['ctype'].isin(['mg1', 'mg2']), 'class'] = 'mg'
df_result.loc[df_result['ctype'].isin(['lg', 'lf']), 'class'] = 'output'
print(f"Class counts:\n{df_result[['class','seg']].drop_duplicates().groupby('class').count()}")

In [ ]:
# Spine density estimates (spines per µm dendritic length per depth bin)
density_sg = np.asarray([20, 20, 20, 20, 20, 20]) / 10
density_mg = np.asarray([53.000000, 95.647059, 113.352941, 113.388889, 88.187500, 50.958333]) / 10
density_out = np.asarray([38.647059, 55.360000, 64.944444, 74.555556, 63.800000, 70.000000]) / 10

depth_order = ['d0', 'd1', 'd2', 'd3', 'd4', 'd5']
density_lookup = {
    'sg': dict(zip(depth_order, density_sg)),
    'mg': dict(zip(depth_order, density_mg)),
    'output': dict(zip(depth_order, density_out))
}

def get_density(row):
    return density_lookup[row['class']][row['labels']]

df_result['sp_density'] = df_result.apply(get_density, axis=1)
df_result['sp_total'] = df_result['dx_total'] * df_result['sp_density']

print("Spine count summary by class:")
print(df_result[['class', 'sp_total']].groupby('class').sum())

In [ ]:
# Save spine count results for use in Analyses_published.ipynb
save_path = DATA_ROOT / 'data_processed_published/df_spine_counts.csv'
df_result.to_csv(save_path, index=False)
print(f"Saved spine count data to: {save_path}")
print(f"Shape: {df_result.shape}")
print(df_result.head())